# HealthDrive Triage Assistant: retraining, held-out evaluation and Hugging Face release

This notebook retrains the HealthDrive triage adapter with the same settings as the original project, then measures the base model and the fine-tuned model on **200 questions the model never sees in training**, and publishes the adapter, the real training log and the real scores to Hugging Face.

Run it on a **T4 GPU** (Runtime > Change runtime type > T4 GPU), then Runtime > Run all. Before running, add your own Hugging Face **Write** token as a Colab secret named `HF_TOKEN` (key icon on the left) and turn on notebook access.

Results of the November 2026 run (200 held-out questions, no overlap with training): base BLEU 16.30, ROUGE-1 0.446, ROUGE-L 0.394; fine-tuned BLEU 40.59, ROUGE-1 0.681, ROUGE-L 0.626. Full files: https://huggingface.co/AgbajeCity/healthdrive-triage-llama3-8b-lora

In [ ]:
%%capture
!pip install unsloth
!pip install evaluate rouge_score huggingface_hub

In [ ]:
# 1. Log in to Hugging Face with the HF_TOKEN secret (key icon on the left; you add the token there yourself)
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
HF_USER = api.whoami()["name"]
REPO_ID = f"{HF_USER}/healthdrive-triage-llama3-8b-lora"
print("Will publish to:", REPO_ID)

In [ ]:
# 2. Data: same cleaning as the original project. Train on the same first 2,000 rows; hold out the next 200 rows for testing.
from datasets import load_dataset
raw = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")
n0 = len(raw)
clean = raw.filter(lambda x: x.get("input") and x.get("output") and len(x["input"].strip()) > 0 and len(x["output"].strip()) > 0)
print(f"Removed {n0 - len(clean)} invalid rows; {len(clean)} remain.")
train_ds = clean.select(range(2000))
test_ds = clean.select(range(2000, 2200))
train_q = set(train_ds["input"])
overlap = sum(q in train_q for q in test_ds["input"])
print(f"Train: {len(train_ds)}  Held-out test: {len(test_ds)}  Test questions also in train: {overlap}")

In [ ]:
# 3. Base model (same as the original project)
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=max_seq_length, dtype=None, load_in_4bit=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
prompt_template = """Below is a medical question. Write a clinical and accurate answer.

### Question:
{}

### Answer:
{}"""
EOS_TOKEN = tokenizer.eos_token

In [ ]:
# 4. Evaluation helper: greedy decoding, batched, scores only the newly generated text
import evaluate, json, numpy as np
bleu = evaluate.load("bleu"); rouge = evaluate.load("rouge")
def generate_answers(m, questions, bs=8, max_new_tokens=64):
    FastLanguageModel.for_inference(m)
    tokenizer.padding_side = "left"
    preds = []
    for i in range(0, len(questions), bs):
        batch = [prompt_template.format(q, "") for q in questions[i:i+bs]]
        enc = tokenizer(batch, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
        for row in out[:, enc["input_ids"].shape[1]:]:
            t = tokenizer.decode(row, skip_special_tokens=True)
            preds.append(t.split("### Question")[0].strip())
        print(f"  {min(i+bs, len(questions))}/{len(questions)}", end="\r")
    return preds
def score(preds, refs):
    b = bleu.compute(predictions=preds, references=[[r] for r in refs])["bleu"] * 100
    r = rouge.compute(predictions=preds, references=refs)
    return {"bleu": round(b, 2), "rouge1": round(r["rouge1"], 3), "rougeL": round(r["rougeL"], 3)}
test_q = list(test_ds["input"]); test_ref = list(test_ds["output"])

In [ ]:
# 5. Score the BASE model on the 200 held-out questions
base_preds = generate_answers(model, test_q)
base_scores = score(base_preds, test_ref)
print("\nBase model:", base_scores)

In [ ]:
# 6. Add LoRA adapters and train (same settings as the original Run 3)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407, use_rslora=False, loftq_config=None)
FastLanguageModel.for_training(model)
tokenizer.padding_side = "right"
def fmt(ex):
    return {"text": [prompt_template.format(i, o) + EOS_TOKEN for i, o in zip(ex["input"], ex["output"])]}
mapped = train_ds.map(fmt, batched=True)
from unsloth import is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=mapped,
    dataset_text_field="text", max_seq_length=max_seq_length, dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1, gradient_accumulation_steps=8, warmup_steps=5,
        num_train_epochs=1, learning_rate=2e-4,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir="outputs", report_to="none"))
stats = trainer.train()
loss_log = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
print("Final logged loss:", loss_log[-1], " Mean train loss:", round(stats.training_loss, 4))

In [ ]:
# 7. Score the FINE-TUNED model on the same 200 held-out questions
tuned_preds = generate_answers(model, test_q)
tuned_scores = score(tuned_preds, test_ref)
print("\nBase:", base_scores, "\nFine-tuned:", tuned_scores)

In [ ]:
# 8. Save the real results and charts
import os, csv, matplotlib.pyplot as plt
os.makedirs("evaluation", exist_ok=True)
results = {"test_set": "200 held-out rows (2000-2199) of medalpaca/medical_meadow_medical_flashcards after cleaning; none used in training",
           "decoding": "greedy, max 64 new tokens", "base": base_scores, "fine_tuned": tuned_scores,
           "train_rows": 2000, "epochs": 1, "steps": loss_log[-1][0], "final_logged_loss": loss_log[-1][1],
           "mean_train_loss": round(stats.training_loss, 4)}
json.dump(results, open("evaluation/metrics.json", "w"), indent=2)
with open("evaluation/training_loss.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["step", "loss"]); w.writerows(loss_log)
with open("evaluation/heldout_predictions.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["question", "reference", "base_answer", "fine_tuned_answer"])
    w.writerows(zip(test_q, test_ref, base_preds, tuned_preds))
plt.figure(figsize=(9, 5)); plt.plot(*zip(*loss_log), marker="o", color="#2563eb")
plt.title("HealthDrive fine-tuning: training loss (real log)"); plt.xlabel("Step"); plt.ylabel("Loss"); plt.grid(alpha=.4)
plt.savefig("evaluation/training_loss.png", dpi=200, bbox_inches="tight"); plt.close()
labels = ["BLEU", "ROUGE-1 (x100)", "ROUGE-L (x100)"]
b = [base_scores["bleu"], base_scores["rouge1"]*100, base_scores["rougeL"]*100]
t = [tuned_scores["bleu"], tuned_scores["rouge1"]*100, tuned_scores["rougeL"]*100]
x = np.arange(3); plt.figure(figsize=(9, 5))
plt.bar(x-.18, b, .36, label="Base Llama-3-8B", color="#94a3b8"); plt.bar(x+.18, t, .36, label="HealthDrive fine-tuned", color="#2563eb")
for i in range(3):
    plt.text(x[i]-.18, b[i]+.5, f"{b[i]:.1f}", ha="center"); plt.text(x[i]+.18, t[i]+.5, f"{t[i]:.1f}", ha="center")
plt.xticks(x, labels); plt.title("Held-out test (200 unseen questions)"); plt.legend()
plt.savefig("evaluation/heldout_scores.png", dpi=200, bbox_inches="tight"); plt.close()
print(json.dumps(results, indent=2))

In [ ]:
# 9. Publish the adapter, tokenizer, model card and evaluation files to Hugging Face
api.create_repo(REPO_ID, exist_ok=True)
model.push_to_hub(REPO_ID); tokenizer.push_to_hub(REPO_ID)
card = f"""---
base_model: unsloth/llama-3-8b-bnb-4bit
library_name: peft
license: llama3
datasets:
- medalpaca/medical_meadow_medical_flashcards
language:
- en
tags:
- medical
- triage
- qlora
- unsloth
---
# HealthDrive Triage Assistant (LoRA adapter for Llama-3-8B)

A QLoRA adapter (rank 16) that tunes Meta's Llama-3-8B (4-bit, via Unsloth) to answer short clinical questions, built by HealthDrive Technologies LTD (Kigali, Rwanda) as the triage layer of a phone service for rural primary care.

## Training
- Data: the first 2,000 cleaned rows of `medalpaca/medical_meadow_medical_flashcards` (empty rows removed)
- One epoch, learning rate 2e-4, effective batch size 8 (1 x 8 gradient accumulation), AdamW 8-bit, linear schedule, seed 3407
- {results['steps']} steps; final logged loss {results['final_logged_loss']:.3f}; mean training loss {results['mean_train_loss']}

## Held-out evaluation
200 rows (2000 to 2199) of the same dataset, none seen in training. Greedy decoding, up to 64 new tokens.

| Model | BLEU | ROUGE-1 | ROUGE-L |
| --- | --- | --- | --- |
| Base Llama-3-8B | {base_scores['bleu']} | {base_scores['rouge1']} | {base_scores['rougeL']} |
| HealthDrive fine-tuned | {tuned_scores['bleu']} | {tuned_scores['rouge1']} | {tuned_scores['rougeL']} |

Every prediction is in `evaluation/heldout_predictions.csv`.

## Prompt format
```
Below is a medical question. Write a clinical and accurate answer.

### Question:
<question>

### Answer:
```

## Use
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained("{REPO_ID}", max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
```

## Limits
Research prototype for educational purposes. English only. Trained on medical flashcards, not on patient conversations. It does not diagnose, prescribe or replace a clinician or emergency service.

Code: https://github.com/AgbajeCity/HealthDrive-Triage-Chatbot (MIT). Base model use is subject to the Meta Llama 3 Community License.
"""
open("README.md", "w").write(card)
api.upload_file(path_or_fileobj="README.md", path_in_repo="README.md", repo_id=REPO_ID)
api.upload_folder(folder_path="evaluation", path_in_repo="evaluation", repo_id=REPO_ID)
print("Published: https://huggingface.co/" + REPO_ID)